# Slide Exercise 02: SBERT Semantic Movie Recommender

This is the refined version of `SBERT_MovieRecommender.ipynb`.

Learning objectives:
- Compare lexical TF-IDF similarity with semantic embedding similarity.
- Use `all-MiniLM-L6-v2` when available.
- Keep the exercise runnable with a TF-IDF fallback.

Main functions used:
- `SentenceTransformer(...)`: loads a pretrained sentence embedding model.
- `model.encode(...)`: converts descriptions into dense semantic vectors.
- `cosine_similarity(...)`: compares embedding vectors.
- `try/except`: keeps optional model code from breaking the notebook.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


Build a text field that reads like a short movie profile.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

movies["profile_text"] = movies["title"] + ". " + movies["description"] + " Genres: " + movies["genres"].str.replace("|", ", ", regex=False)

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["profile_text"])
tfidf_similarity = cosine_similarity(tfidf_matrix)


Try SBERT. If it is missing or cannot load the model, use the TF-IDF matrix instead.


In [ ]:
model_name = "TF-IDF fallback"
semantic_similarity = tfidf_similarity
semantic_vectors = tfidf_matrix

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    semantic_vectors = model.encode(movies["profile_text"].tolist(), show_progress_bar=False)
    semantic_similarity = cosine_similarity(semantic_vectors)
    model_name = "SBERT all-MiniLM-L6-v2"
except Exception as exc:
    print("SBERT is optional for this exercise. Continuing with TF-IDF fallback.")
    print(type(exc).__name__, str(exc)[:160])

model_name


Use one ranking function for both lexical and semantic similarities.


In [ ]:
def recommend(title, similarity_matrix, label, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    ranked = similarity_matrix[idx].argsort()[::-1]
    rows = []
    for other_idx in ranked:
        if other_idx == idx:
            continue
        rows.append({
            "method": label,
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "score": round(float(similarity_matrix[idx, other_idx]), 3),
        })
        if len(rows) == n:
            break
    return pd.DataFrame(rows)

pd.concat([
    recommend("Gravity", tfidf_similarity, "TF-IDF"),
    recommend("Gravity", semantic_similarity, model_name),
], ignore_index=True)


Interpretation:

TF-IDF rewards shared words. SBERT, when available, can connect descriptions that express similar meaning with different words.

Student task:
1. Compare recommendations for `Titanic`.
2. Add a new movie with a description that uses different words for a similar idea.
